##설치 및 요약


In [1]:
## ==== 0) (필요시) 설치 ====

!pip -q install -U transformers datasets torch pandas sentencepiece tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 13.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dask-cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.
cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0;

In [4]:

# =========================
# 1) 라이브러리 임포트 & 디바이스
# =========================
import os
import pandas as pd
from tqdm import tqdm
from transformers import pipeline, AutoTokenizer
import torch

# =========================
# 2) CSV 업로드 (Colab 파일 업로드 위젯)
#    - 업로드 후 아래 'INPUT_CSV' 파일명을 맞게 적어주세요.
# =========================
from google.colab import files
uploaded = files.upload()  # 예: naver_finance_top10.csv 업로드

# 업로드된 파일명 자동 추출 (여러 개 올리면 맨 첫 번째 사용)
INPUT_CSV = list(uploaded.keys())[0]
OUTPUT_CSV = INPUT_CSV.replace(".csv", "_summarized.csv")

# =========================
# 3) 모델 선택
#    - 한국어 기사 → KoBART
#    - 영어 기사 → BART-CNN
# =========================
MODEL_ID = "gogamza/kobart-summarization"     # 한국어
# MODEL_ID = "facebook/bart-large-cnn"        # 영어(CNN/DailyMail 기반)

# =========================
# 4) 요약 대상 컬럼 후보 설정
#    - 본문 컬럼 추정: body/content/text 중 하나
#    - 제목 컬럼(있으면 힌트로 앞에 붙여 품질↑): title_from_list/title/headline
# =========================
TEXT_COL_CANDIDATES = ["body", "content", "text"]
TITLE_COL_CANDIDATES = ["title_from_list", "title", "headline"]

# =========================
# 5) 디바이스 설정
# =========================
device = 0 if torch.cuda.is_available() else -1
print("Using device:", "GPU" if device == 0 else "CPU")

# =========================
# 6) 데이터 로드 & 컬럼 자동탐지
# =========================
df = pd.read_csv(INPUT_CSV)
cols_lower = {c.lower(): c for c in df.columns}

def pick_col(cands, default=None):
    for c in cands:
        if c in cols_lower:
            return cols_lower[c]
    return default

TEXT_COL  = pick_col(TEXT_COL_CANDIDATES, default=df.columns[-1])  # 그래도 없으면 마지막 컬럼 사용
TITLE_COL = pick_col(TITLE_COL_CANDIDATES, default=None)

# 빈값 대비
df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("")
if TITLE_COL is not None:
    df[TITLE_COL] = df[TITLE_COL].astype(str).fillna("")

print("Detected TEXT_COL:", TEXT_COL)
print("Detected TITLE_COL:", TITLE_COL)

# =========================
# 7) 요약 파이프라인 준비
# =========================
summarizer = pipeline(
    task="summarization",
    model=MODEL_ID,
    device=device
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 모델 최대 토큰 길이
model_max = getattr(tokenizer, "model_max_length", 1024)
max_input_tokens = min(model_max, 1024)

# 긴 텍스트를 토큰 기준으로 청크 분할
def chunk_by_tokens(text, max_tokens=1024, overlap_tokens=50):
    text = (text or "").strip()
    if not text:
        return []
    enc = tokenizer(text, return_tensors=None, add_special_tokens=False)
    ids = enc["input_ids"]
    chunks = []
    start = 0
    while start < len(ids):
        end = min(start + max_tokens, len(ids))
        piece_ids = ids[start:end]
        chunk_text = tokenizer.decode(piece_ids, skip_special_tokens=True)
        chunks.append(chunk_text)
        if end == len(ids): break
        start = max(0, end - overlap_tokens)
    return chunks

# 청크 요약 → 합치기 → (선택) 한 번 더 압축
def summarize_text(text, max_summary_len=150, min_summary_len=30):
    text = (text or "").strip()
    if not text:
        return ""

    chunks = chunk_by_tokens(text, max_tokens=max_input_tokens-32, overlap_tokens=40)
    # 짧으면 1회 요약
    if len(chunks) <= 1:
        try:
            out = summarizer(
                text,
                max_length=max_summary_len,
                min_length=min_summary_len,
                do_sample=False
            )[0]["summary_text"]
            return out.strip()
        except Exception:
            return ""

    # 길면 청크별 요약 후 합치기
    partial_summaries = []
    for ck in chunks:
        try:
            s = summarizer(
                ck,
                max_length=max_summary_len,
                min_length=min_summary_len,
                do_sample=False
            )[0]["summary_text"]
            partial_summaries.append(s.strip())
        except Exception:
            continue
    if not partial_summaries:
        return ""

    joined = "\n".join(partial_summaries)
    # 최종 압축 요약 (선택)
    try:
        final = summarizer(
            joined,
            max_length=max_summary_len,
            min_length=min_summary_len,
            do_sample=False
        )[0]["summary_text"]
        return final.strip()
    except Exception:
        return joined.strip()

# =========================
# 8) 전체 데이터 요약 실행
# =========================
summaries = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Summarizing"):
    base_text = row[TEXT_COL]
    if TITLE_COL is not None and str(row[TITLE_COL]).strip():
        # 제목을 앞에 붙여 힌트 제공 → 요약 품질 개선
        text = f"{row[TITLE_COL]}\n{base_text}"
    else:
        text = base_text

    try:
        s = summarize_text(text)
    except Exception:
        s = ""
    summaries.append(s)

df["summary"] = summaries

# =========================
# 9) 저장 & 다운로드
# =========================
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved: {OUTPUT_CSV}  (rows={len(df)})")
print("Columns:", list(df.columns))

from google.colab import files
files.download(OUTPUT_CSV)


Saving naver_finance_top10.csv to naver_finance_top10.csv
Using device: CPU
Detected TEXT_COL: body
Detected TITLE_COL: title_from_list


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
Device set to use cpu
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
Summarizing:  20%|██        | 2/10 [00:15<01:01,  7.65s/it]Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more informa

Saved: naver_finance_top10_summarized.csv  (rows=10)
Columns: ['rank', 'title_from_list', 'url', 'body', 'summary']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##분석

In [6]:
# 한국어 감성 모델 (NSMC 파인튜닝) 예시
SENT_MODEL = "nlptown/bert-base-multilingual-uncased-sentiment"  # binary pos/neg
# 다국어 NER (wikiann 기반) – 한국어 포함
NER_MODEL  = "Davlan/xlm-roberta-base-ner-hrl"

# =========================
# 1) 임포트 & 업로드
# =========================
import re
import pandas as pd
from tqdm import tqdm
from transformers import pipeline

from google.colab import files
uploaded = files.upload()  # 예: naver_finance_top10_summarized.csv
INPUT_CSV = list(uploaded.keys())[0]
OUTPUT_CSV = INPUT_CSV.replace(".csv", "_structured.csv")

df = pd.read_csv(INPUT_CSV)
cols = {c.lower(): c for c in df.columns}
TEXT_COL = cols.get("summary") or cols.get("body") or cols.get("content") or list(df.columns)[-1]
TITLE_COL = cols.get("title") or cols.get("title_from_list")

df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("")
if TITLE_COL: df[TITLE_COL] = df[TITLE_COL].astype(str).fillna("")

print("TEXT_COL =", TEXT_COL, "| TITLE_COL =", TITLE_COL)

# =========================
# 2) 파이프라인 준비 (Transformers)
# =========================
ner = pipeline("token-classification", model=NER_MODEL, aggregation_strategy="simple")
senti = pipeline("text-classification", model=SENT_MODEL)

# =========================
# 3) 규칙(키워드, 단위, 패턴)
# =========================
METRIC_KEYWORDS = [
    "매출", "영업이익", "순이익", "주가", "매출액", "매출총이익", "판매량", "출하량", "수출", "수주",
    "점유율", "배당", "가이던스", "수익률", "금리", "원가", "마진"
]
METRIC_KEYWORDS += [
    "수익률", "지수", "가격", "금값", "환율", "달러", "ETF", "코스피", "코스닥", "닛케이",
    "WTI", "유가", "비트코인", "시총", "물가", "인플레이션"
]

UP_WORDS   = ["상승", "증가", "확대", "개선", "반등", "강세", "인상", "급등", "호조"]
DOWN_WORDS = ["하락", "감소", "축소", "악화", "급락", "약세", "인하", "둔화", "부진"]
UP_WORDS += ["기록", "경신", "돌파", "상승세", "호조세", "강세", "최고치", "사상 최고"]
DOWN_WORDS += ["추락", "하락세", "약세", "최저치", "급락세"]

# 숫자/단위 패턴
PCT_R = re.compile(r'[-+]?\d+(?:\.\d+)?\s*%')
KRW_R = re.compile(r'(?:(?:\d{1,3}(?:,\d{3})+|\d+(?:\.\d+)?)\s*(조|억|만)?\s*원)')
RAWNUM_R = re.compile(r'[-+]?\d+(?:\.\d+)?')

# 주변 윈도우에서 대상/규모를 뽑기 위한 간단 헬퍼
def window(text, idx, w=30):
    s = max(0, idx - w); e = min(len(text), idx + w)
    return text[s:e]

# =========================
# 4) 한 문서(요약문) → 구조화
# =========================
def extract_entities(text):
    orgs = []
    try:
        for ent in ner(text):
            if ent.get("entity_group") in ("ORG","PER","MISC"):  # 기업/브랜드가 ORG/MISC로 나오는 경우 多
                orgs.append(ent["word"])
    except Exception:
        pass
    # 중복 제거(길이 긴 것 우선)
    orgs = sorted(set(orgs), key=lambda x: (-len(x), x))
    return orgs[:3]  # 상위 3개까지만

def extract_metric(text):
    hits = [m for m in METRIC_KEYWORDS if m in text]
    # 우선 등장하는 순서대로
    return hits[0] if hits else ""

def extract_direction_and_magnitude(text):
    # 기본 방향
    direction = ""
    mag = ""
    unit = ""

    # 각 방향 키워드 위치 검색
    for w in UP_WORDS + DOWN_WORDS:
        for m in re.finditer(re.escape(w), text):
            ctx = window(text, m.start(), w=40)
            # % 또는 금액을 근처에서 우선 탐색
            pct = PCT_R.search(ctx)
            krw = KRW_R.search(ctx)
            raw = RAWNUM_R.search(ctx)

            if w in UP_WORDS:
                direction = "up"
            elif w in DOWN_WORDS:
                direction = "down"

            if pct:
                mag = pct.group().replace(" ", "")
                unit = "%"
            elif krw:
                mag = krw.group().replace(" ", "")
                unit = "KRW"
            elif raw:
                # 단위 없는 숫자: 일단 숫자만
                mag = raw.group()
                unit = ""

            if direction:
                return direction, mag, unit

    return "", "", ""

def classify_sentiment(text):
    try:
        out = senti(text[:512])[0]  # 길면 잘라서
        # 모델 라벨이 'positive/negative' 혹은 'LABEL_0/1' 형태일 수 있음
        label = out["label"].lower()
        if "pos" in label or "positive" in label:
            return "positive", float(out.get("score", 0.0))
        elif "neg" in label or "negative" in label:
            return "negative", float(out.get("score", 0.0))
        else:
            return label, float(out.get("score", 0.0))
    except Exception:
        return "", 0.0

def structure_one_row(row):
    text = row[TEXT_COL]
    if TITLE_COL and str(row[TITLE_COL]).strip():
        text = f"{row[TITLE_COL]}\n{text}"

    entities = extract_entities(text)
    metric   = extract_metric(text)
    direction, magnitude, unit = extract_direction_and_magnitude(text)
    tone, tone_score = classify_sentiment(text)

    return {
        "entity": entities[0] if entities else "",
        "entities_all": "; ".join(entities),
        "metric": metric,
        "direction": direction,         # up/down/""
        "magnitude": magnitude,         # e.g., "12.3%", "3,000억원"
        "sentiment_score": tone_score,
        "summary": row[TEXT_COL]
    }

# =========================
# 5) 전체 적용
# =========================
records = []
for _, r in tqdm(df.iterrows(), total=len(df), desc="Structuring"):
    rec = structure_one_row(r)
    records.append(rec)

out = pd.DataFrame(records)
out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved: {OUTPUT_CSV}")

from google.colab import files
files.download(OUTPUT_CSV)


Saving naver_finance_top10_summarized.csv to naver_finance_top10_summarized (2).csv
TEXT_COL = summary | TITLE_COL = title_from_list


Device set to use cpu


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cpu
Structuring: 100%|██████████| 10/10 [00:13<00:00,  1.31s/it]

Saved: naver_finance_top10_summarized (2)_structured.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## kiwi

In [2]:
# =========================
# 0) 설치 (Colab)
# =========================
!pip -q install -U transformers pandas sentencepiece tqdm kiwipiepy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 33.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 116.7 MB/s eta 0:00:00


In [10]:
# =========================
# 1) 임포트 & 업로드
# =========================
import re
import pandas as pd
from tqdm import tqdm
from transformers import pipeline
from google.colab import files

# 형태소 분석기 (Kiwi)
from kiwipiepy import Kiwi
kiwi = Kiwi()  # 기본 사전 로드

uploaded = files.upload()  # 예: naver_finance_top10_summarized.csv
INPUT_CSV = list(uploaded.keys())[0]
OUTPUT_CSV = INPUT_CSV.replace(".csv", "_structured.csv")

df = pd.read_csv(INPUT_CSV)
cols = {c.lower(): c for c in df.columns}
TEXT_COL = cols.get("summary") or cols.get("body") or cols.get("content") or list(df.columns)[-1]
TITLE_COL = cols.get("title") or cols.get("title_from_list")

df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("")
if TITLE_COL: df[TITLE_COL] = df[TITLE_COL].astype(str).fillna("")

print("TEXT_COL =", TEXT_COL, "| TITLE_COL =", TITLE_COL)

# =========================
# 2) 파이프라인 (Transformers)
#    감성: 다국어 별점 → pos/neg/neutral 매핑
#    NER: 다국어 NER
# =========================
SENT_MODEL = "nlptown/bert-base-multilingual-uncased-sentiment"
NER_MODEL  = "Davlan/xlm-roberta-base-ner-hrl"

ner   = pipeline("token-classification", model=NER_MODEL, aggregation_strategy="simple")
senti = pipeline("text-classification",   model=SENT_MODEL)

# =========================
# 3) 규칙(키워드, 패턴)
# =========================
METRIC_KEYWORDS = [
    "매출", "영업이익", "순이익", "주가", "매출액", "매출총이익", "판매량", "출하량", "수출", "수주",
    "점유율", "배당", "가이던스", "수익률", "금리", "원가", "마진", "실적", "이익률", "매출총이익률"
]

UP_WORDS   = ["상승", "증가", "확대", "개선", "반등", "강세", "인상", "급등", "호조", "늘어"]
DOWN_WORDS = ["하락", "감소", "축소", "악화", "급락", "약세", "인하", "둔화", "부진", "줄어"]

# 숫자/단위 패턴
PCT_R    = re.compile(r'[-+]?\d+(?:\.\d+)?\s*%')
KRW_R    = re.compile(r'(?:(?:\d{1,3}(?:,\d{3})+|\d+(?:\.\d+)?)\s*(?:조|억|만)?\s*원)')
RAWNUM_R = re.compile(r'[-+]?\d+(?:\.\d+)?')

def window(text, idx, w=30):
    s = max(0, idx - w); e = min(len(text), idx + w)
    return text[s:e]

# =========================
# 4) Kiwi 기반 추출 로직
# =========================
# (A) 기업/브랜드 엔티티
def extract_entities(text):
    orgs = []
    try:
        for ent in ner(text):
            if ent.get("entity_group") in ("ORG","PER","MISC"):  # 기업/브랜드가 ORG/MISC로 나오는 경우 多
                orgs.append(ent["word"])
    except Exception:
        pass
    orgs = sorted(set(orgs), key=lambda x: (-len(x), x))  # 길이 우선
    return orgs[:3]

# (B) Kiwi로 metric 후보 더 정확히 찾기
# ===== 교체: Kiwi 토큰을 안전하게 순회하는 유틸 =====
def iter_kiwi_tokens(text):
    """
    kiwi.tokenize(text) 결과를 (form, tag, start, length) 튜플 리스트로 표준화.
    환경에 따라 Token 객체/튜플/문자열 등으로 나올 수 있어 안전하게 처리.
    """
    tokens_std = []
    try:
        toks = kiwi.tokenize(text)   # analyze 대신 tokenize로 단순 토큰화
    except Exception:
        toks = []

    for t in toks:
        # 가능한 속성들에서 안전하게 뽑기
        if hasattr(t, "form") or hasattr(t, "tag"):
            form   = getattr(t, "form", "")
            tag    = getattr(t, "tag", "")
            start  = getattr(t, "start", None)
            length = getattr(t, "length", None)
        elif isinstance(t, tuple):
            # (form, tag, start, length) 혹은 (form, tag) 등 형태 가정
            form   = t[0] if len(t) > 0 else ""
            tag    = t[1] if len(t) > 1 else ""
            start  = t[2] if len(t) > 2 else None
            length = t[3] if len(t) > 3 else None
        else:
            # 문자열 등 예상치 못한 타입
            form, tag, start, length = str(t), "", None, None

        # start/length가 없으면 대략 추정 (최초 등장 위치)
        if (start is None or length is None) and form:
            pos = text.find(form)
            if pos >= 0:
                start, length = pos, len(form)

        tokens_std.append((form, tag, start, length))
    return tokens_std


# ===== 교체: Kiwi 기반 metric 추출 함수 =====
def extract_metric_with_kiwi(text):
    cues = UP_WORDS + DOWN_WORDS

    # 상승/하락 cue 위치
    cue_spans = []
    for w in cues:
        for m in re.finditer(re.escape(w), text):
            cue_spans.append((m.start(), m.end(), w))
    if not cue_spans:
        # cue가 없으면 사전 키워드로 폴백
        for k in METRIC_KEYWORDS:
            if k in text:
                return k
        return ""

    tokens = iter_kiwi_tokens(text)
    if not tokens:
        for k in METRIC_KEYWORDS:
            if k in text:
                return k
        return ""

    # cue 주변 윈도우에서 명사 후보 수집
    from collections import Counter
    candidates = []
    for cs, ce, _ in cue_spans:
        ws, we = max(0, cs - 40), min(len(text), ce + 40)
        for form, tag, start, length in tokens:
            if not form or start is None or length is None:
                continue
            # 명사/외국어/한자
            if not (str(tag).startswith("NN") or tag in ("SL", "SH")):
                continue
            if ws <= start and (start + length) <= we:
                form = form.strip()
                if len(form) >= 2:
                    candidates.append(form)

    if candidates:
        # 1) 금융 지표 사전과 교집합 우선
        dict_hits = [c for c in candidates if any(k in c for k in METRIC_KEYWORDS)]
        if dict_hits:
            cnt = Counter(dict_hits)
            best = sorted(cnt.items(), key=lambda x: (-x[1], -len(x[0]), x[0]))[0][0]
            # 대표 키워드로 정규화
            for k in METRIC_KEYWORDS:
                if k in best:
                    return k
            return best

        # 2) 없으면 빈도/길이 기준 최적 후보
        cnt = Counter(candidates)
        best = sorted(cnt.items(), key=lambda x: (-x[1], -len(x[0]), x[0]))[0][0]
        for k in METRIC_KEYWORDS:
            if k in best:
                return k
        return best

    # 최종 폴백
    for k in METRIC_KEYWORDS:
        if k in text:
            return k
    return ""


# (C) 방향/규모
def extract_direction_and_magnitude(text):
    direction = ""
    mag = ""

    for w in UP_WORDS + DOWN_WORDS:
        for m in re.finditer(re.escape(w), text):
            ctx = window(text, m.start(), w=50)
            pct = PCT_R.search(ctx)
            krw = KRW_R.search(ctx)
            raw = RAWNUM_R.search(ctx)

            if w in UP_WORDS:
                direction = "up"
            elif w in DOWN_WORDS:
                direction = "down"

            if pct:
                mag = pct.group().replace(" ", "")
            elif krw:
                mag = krw.group().replace(" ", "")
            elif raw:
                mag = raw.group()

            if direction:
                return direction, mag

    return "", ""

# (D) 감성 (별점 → pos/neg/neutral)
def classify_sentiment(text):
    try:
        out = senti(text[:512])[0]  # {'label': '4 stars', 'score': ...}
        label = out["label"]
        try:
            stars = int(label.split()[0])
        except:
            stars = 3
        if stars >= 4: return "positive", float(out.get("score", 0.0))
        if stars <= 2: return "negative", float(out.get("score", 0.0))
        return "neutral", float(out.get("score", 0.0))
    except Exception:
        return "", 0.0

# =========================
# 5) 한 행 처리
# =========================
def structure_one_row(row):
    text = row[TEXT_COL]
    if TITLE_COL and str(row[TITLE_COL]).strip():
        text = f"{row[TITLE_COL]}\n{text}"

    entities = extract_entities(text)
    # ✅ Kiwi로 metric 우선
    metric = extract_metric_with_kiwi(text)
    direction, magnitude = extract_direction_and_magnitude(text)
    tone, tone_score = classify_sentiment(text)

    return {
        "entity": (entities[0] if entities else ""),
        "entities_all": "; ".join(entities),
        "metric": metric,               # Kiwi 기반
        "direction": direction,         # up/down/""
        "magnitude": magnitude,         # e.g., "12.3%", "3,000억원"
        # ✅ unit 열 제거
        "sentiment": tone,              # positive/negative/neutral
        "sentiment_score": tone_score,
        "summary": row[TEXT_COL],
    }

# =========================
# 6) 전체 적용 & 저장
# =========================
records = []
for _, r in tqdm(df.iterrows(), total=len(df), desc="Structuring"):
    rec = structure_one_row(r)
    records.append(rec)

out = pd.DataFrame(records)
out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved: {OUTPUT_CSV}")

files.download(OUTPUT_CSV)


Saving naver_finance_top10_summarized.csv to naver_finance_top10_summarized (5).csv
TEXT_COL = summary | TITLE_COL = title_from_list


Device set to use cpu
Device set to use cpu
Structuring: 100%|██████████| 10/10 [00:15<00:00,  1.50s/it]

Saved: naver_finance_top10_summarized (5)_structured.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##관계추출 추가

In [4]:
!pip -q install -U transformers pandas kiwipiepy sentencepiece tqdm

import re, pandas as pd
from tqdm import tqdm
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
from kiwipiepy import Kiwi
from google.colab import files

# 1) 데이터 업로드
uploaded = files.upload()  # naver_finance_top10_summarized.csv
INPUT = list(uploaded.keys())[0]
OUT = INPUT.replace(".csv", "_re_structured.csv")
df = pd.read_csv(INPUT)
cols = {c.lower(): c for c in df.columns}
TEXT_COL = cols.get("summary") or cols.get("body") or cols.get("content") or list(df.columns)[-1]
TITLE_COL = cols.get("title") or cols.get("title_from_list")
df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("")
if TITLE_COL: df[TITLE_COL] = df[TITLE_COL].astype(str).fillna("")

# 2) 도구 준비
device = 0 if False else -1   # GPU 쓰려면: True로 바꾸고, 런타임 GPU 설정 후 0
ner = pipeline("token-classification", model="Davlan/xlm-roberta-base-ner-hrl",
               aggregation_strategy="simple", device=0)
# 제로샷 NLI (멀티링구얼)
nli_model = "joeddav/xlm-roberta-large-xnli"
nli = pipeline("text-classification", model=nli_model, tokenizer=nli_model, device=0)

kiwi = Kiwi()

# 3) 후보 추출
FIN_METRICS = [
    "주가","지수","수익률","가격","금","금값","금 가격","유가","WTI","환율",
    "매출","영업이익","순이익","마진","이익률","배당","점유율","가이던스","시총"
]
UP_CUES   = ["상승","증가","반등","강세","급등","호조","경신","돌파","최고치","신고가","기록"]
DOWN_CUES = ["하락","감소","약세","급락","추락","둔화","부진","최저치","신저가"]

PCT = re.compile(r'[-+]?\d+(?:\.\d+)?\s*%')
NUM = re.compile(r'[-+]?\d+(?:,\d{3})*(?:\.\d+)?')
CUR = re.compile(r'(?:달러|원|위안|엔|유로|포인트)')

def find_entities(text):
    ents=[]
    try:
        for e in ner(text):
            if e["entity_group"] in ("ORG","MISC","PER","LOC"):
                ents.append(e["word"])
    except Exception:
        pass
    # 중복 제거(길이↓ 우선 제거)
    uniq=[]
    for w in ents:
        if w not in uniq: uniq.append(w)
    return uniq[:3]

def find_metrics(text):
    # ① 사전 매칭
    hits=[m for m in FIN_METRICS if m in text]
    # ② 수치 주변 명사 (Kiwi)
    toks=kiwi.tokenize(text)
    wins=[]
    for m in re.finditer(r'[-+]?\d+(?:,\d{3})*(?:\.\d+)?\s*(?:%|달러|원|위안|엔|유로|포인트)?', text):
        s,e=m.start(),m.end()
        for t in toks:
            if str(t.tag).startswith("NN") and s-20 <= t.start <= e+20 and len(t.form)>1:
                wins.append(t.form)
    # 사전과 교집합 우선
    for w in wins:
        for k in FIN_METRICS:
            if k in w: hits.append(k)
    # 대표 하나 반환
    return list(dict.fromkeys(hits))[:2] or []

def extract_magnitude(text):
    # % 우선, 없으면 숫자+통화/포인트
    ctx = PCT.search(text)
    if ctx: return ctx.group().replace(" ","")
    m = NUM.search(text)
    if m:
        tail = text[m.end():m.end()+6]
        cur = CUR.search(tail)
        return (m.group() + (cur.group() if cur else "")).strip()
    return ""

# 4) 제로샷 RE: 가설 템플릿으로 엔테일먼트 점수 비교
RELATIONS = {
    "up":   ["{X}가 올랐다", "{X}가 상승했다", "{X}가 최고치를 경신했다", "{X}가 급등했다", "{X}가 강세다"],
    "down": ["{X}가 내렸다", "{X}가 하락했다", "{X}가 최저로 떨어졌다", "{X}가 급락했다", "{X}가 약세다"],
    "record_high": ["{X}가 사상 최고치다", "{X}가 신고가다", "{X}가 역대 최고를 기록했다"],
    "record_low":  ["{X}가 사상 최저치다", "{X}가 신저가다", "{X}가 역대 최저를 기록했다"],
}

def nli_score(premise, hypothesis):
    r = nli({"text": premise, "text_pair": hypothesis}, top_k=None)
    # XNLI 라벨: entailment/neutral/contradiction
    d = {i["label"]: i["score"] for i in r}
    return d.get("ENTAILMENT", d.get("entailment", 0.0))

def infer_relation(text, target):
    best_rel, best_sc = "neutral", 0.0
    for rel, hyps in RELATIONS.items():
        for hyp in hyps:
            h = hyp.format(X=target)
            sc = nli_score(text, h)
            if sc > best_sc:
                best_sc, best_rel = sc, rel
    # up/down/record_* 외엔 neutral
    if best_sc < 0.55:   # 임계값은 상황에 맞게 조정
        return "neutral", best_sc
    # record_*가 up/down에 포함되지 않도록 우선순위 유지
    return best_rel, best_sc

# 5) 행 단위 처리
def process_row(row):
    text = row[TEXT_COL]
    if TITLE_COL and row[TITLE_COL]:
        text = f"{row[TITLE_COL]}\n{text}"

    entities = find_entities(text)              # 기업/브랜드/지명 등
    metrics  = find_metrics(text)               # 주가/지수/수익률/가격/…
    target   = (metrics[0] if metrics else (entities[0] if entities else "")) or ""

    relation, rel_score = infer_relation(text, target if target else "지수")
    magnitude = extract_magnitude(text)

    return {
        "target": target,            # 관계의 대상 (지표/엔티티)
        "relation": relation,        # up/down/record_high/record_low/neutral
        "relation_score": rel_score, # 엔테일먼트 확신도
        "magnitude": magnitude,      # % 또는 숫자+통화/포인트
        "entities_all": "; ".join(entities),
        "metrics_all": "; ".join(metrics),
        "summary": row[TEXT_COL]
    }

out = pd.DataFrame([process_row(r) for _, r in tqdm(df.iterrows(), total=len(df))])
out.to_csv(OUT, index=False, encoding="utf-8-sig")
print("Saved:", OUT)
files.download(OUT)


Saving naver_finance_top10_summarized.csv to naver_finance_top10_summarized (1).csv


Device set to use cuda:0
Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0
100%|██████████| 10/10 [00:06<00:00,  1.43it/s]

Saved: naver_finance_top10_summarized (1)_re_structured.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>